# ARIMA Models

**ARIMA** stands for **A**uto**R**egressive **I**ntegrated **M**oving
**A**verage.  It combines three components:

| Component | Symbol | Meaning |
|-----------|--------|---------|
| AR($p$) | $p$ | Number of autoregressive lags |
| I($d$) | $d$ | Number of differences to achieve stationarity |
| MA($q$) | $q$ | Number of lagged forecast errors |

The full model in backshift notation:

$$
(1 - \phi_1 B - \phi_2 B^2 - \cdots - \phi_p B^p)(1-B)^d y_t = c + (1 + \theta_1 B + \theta_2 B^2 + \cdots + \theta_q B^q)\varepsilon_t
$$

or compactly: $\Phi(B)(1-B)^d y_t = c + \Theta(B)\varepsilon_t$.

This notebook covers:
1. The role of differencing (the "I" in ARIMA)
2. The complete ARIMA modelling workflow
3. Fitting ARIMA with the modern statsmodels API
4. Automatic order selection with `pmdarima.auto_arima`
5. Complete example on real data with residual diagnostics

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA  # NEW API (not arima_model)
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from sklearn.metrics import mean_absolute_error, mean_squared_error

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["lines.linewidth"] = 1.5

DATA_DIR = Path("../data")

---
## 1. The Role of Differencing

AR and MA models require **stationarity**.  Most real-world time series are
non-stationary (they have trends, changing levels, etc.).  The "I" in ARIMA
stands for **Integrated** — it means we **difference** the series $d$ times
to make it stationary.

- First difference: $y_t' = y_t - y_{t-1} = (1 - B) y_t$
- Second difference: $y_t'' = y_t' - y_{t-1}' = (1 - B)^2 y_t$

In practice, $d = 1$ handles most trend-type non-stationarity.  $d = 2$ is
rarely needed.  $d \geq 3$ almost never.

In [ ]:
# Load the daily births dataset
births = pd.read_csv(
    DATA_DIR / "DailyTotalFemaleBirths.csv",
    index_col="Date",
    parse_dates=True,
)
births.columns = ["Births"]

print(f"Shape: {births.shape}")
print(f"Date range: {births.index[0].date()} to {births.index[-1].date()}")
births.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(births["Births"], linewidth=0.8)
axes[0].set_title("Daily Female Births — Original")
axes[0].set_ylabel("Births")

births_diff = births["Births"].diff().dropna()
axes[1].plot(births_diff, linewidth=0.8)
axes[1].set_title("Daily Female Births — First Difference")
axes[1].set_ylabel("Change in Births")
axes[1].axhline(0, color="grey", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

---
## 2. Testing for Stationarity: ADF and KPSS

Two complementary tests:

| Test | Null hypothesis | Reject when... |
|------|----------------|----------------|
| ADF (Augmented Dickey-Fuller) | Series has a unit root (non-stationary) | p-value < 0.05 → stationary |
| KPSS | Series is stationary | p-value < 0.05 → non-stationary |

Using both together reduces the risk of incorrect conclusions.

In [ ]:
def stationarity_tests(series, name=""):
    """Run ADF and KPSS tests and print results."""
    # ADF test
    adf_stat, adf_p, adf_lags, adf_nobs, adf_crit, _ = adfuller(series, autolag="AIC")
    print(f"=== {name} ===")
    print(f"ADF test statistic: {adf_stat:.4f}")
    print(f"ADF p-value:        {adf_p:.6f}")
    adf_conclusion = "Stationary (reject H0)" if adf_p < 0.05 else "Non-stationary (fail to reject H0)"
    print(f"ADF conclusion:     {adf_conclusion}")
    print()

    # KPSS test
    kpss_stat, kpss_p, kpss_lags, kpss_crit = kpss(series, regression="c", nlags="auto")
    print(f"KPSS test statistic: {kpss_stat:.4f}")
    print(f"KPSS p-value:        {kpss_p:.4f}")
    kpss_conclusion = "Non-stationary (reject H0)" if kpss_p < 0.05 else "Stationary (fail to reject H0)"
    print(f"KPSS conclusion:     {kpss_conclusion}")
    print()


stationarity_tests(births["Births"], "Daily Births — Original")

In [ ]:
stationarity_tests(births_diff, "Daily Births — First Difference")

print("The original series is already borderline stationary for this dataset.")
print("We will try both d=0 and d=1 and let the model selection decide.")

---
## 3. The ARIMA Modelling Workflow

The classical Box-Jenkins approach:

1. **Plot the data** — identify if differencing is needed
2. **ADF/KPSS test** — determine $d$
3. **Difference** if needed, then **plot ACF/PACF** of the differenced data
4. Use ACF/PACF patterns to suggest **tentative $p$ and $q$**
5. **Fit the model**, examine the **summary**
6. **Check residuals** — if not white noise, try different orders
7. **Forecast** and evaluate

In [ ]:
# Step 3: ACF/PACF of the (possibly differenced) data
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

plot_acf(births["Births"], lags=30, ax=axes[0, 0], title="ACF — Original")
plot_pacf(births["Births"], lags=30, ax=axes[0, 1], title="PACF — Original", method="ywm")
plot_acf(births_diff, lags=30, ax=axes[1, 0], title="ACF — First Difference")
plot_pacf(births_diff, lags=30, ax=axes[1, 1], title="PACF — First Difference", method="ywm")

plt.tight_layout()
plt.show()

print("Original: PACF suggests a few significant lags — could try AR(p) with small p.")
print("First difference: sharp ACF cutoff might suggest an MA component.")
print("\nWe will try a few candidate models and compare.")

---
## 4. Train / Test Split

In [ ]:
TRAIN_END = 335
train = births.iloc[:TRAIN_END]
test = births.iloc[TRAIN_END:]
HORIZON = len(test)

print(f"Train: {len(train)} observations ({train.index[0].date()} to {train.index[-1].date()})")
print(f"Test:  {len(test)} observations ({test.index[0].date()} to {test.index[-1].date()})")

---
## 5. Fitting ARIMA with the Modern API

The correct import path is:

```python
from statsmodels.tsa.arima.model import ARIMA  # CORRECT (new API)
# NOT: from statsmodels.tsa.arima_model import ARIMA  (old, removed)
```

Key methods:
- `model.fit()` — estimate parameters
- `result.summary()` — full summary with AIC, BIC, coefficients
- `result.get_forecast(steps=h)` — out-of-sample forecast with prediction intervals
- `result.get_forecast(steps=h).summary_frame()` — forecast as a DataFrame with CI

In [ ]:
# Try several candidate models
candidates = [
    (1, 0, 0),
    (2, 0, 0),
    (0, 0, 1),
    (0, 0, 2),
    (1, 0, 1),
    (2, 0, 1),
    (1, 1, 0),
    (1, 1, 1),
    (2, 1, 1),
    (0, 1, 1),
]

results_table = []
for order in candidates:
    try:
        mod = ARIMA(train["Births"], order=order).fit()
        results_table.append({
            "Order": f"ARIMA{order}",
            "AIC": round(mod.aic, 2),
            "BIC": round(mod.bic, 2),
        })
    except Exception as e:
        results_table.append({"Order": f"ARIMA{order}", "AIC": None, "BIC": None})
        print(f"  ARIMA{order} failed: {e}")

comp_df = pd.DataFrame(results_table).sort_values("AIC").reset_index(drop=True)
print(comp_df.to_string(index=False))
print(f"\nBest by AIC: {comp_df.iloc[0]['Order']}")
print(f"Best by BIC: {comp_df.sort_values('BIC').iloc[0]['Order']}")

In [ ]:
# Fit the best model
best_order_str = comp_df.iloc[0]["Order"]
print(f"Fitting {best_order_str} ...")

# Parse order from the best result
best_order = comp_df.sort_values("AIC").iloc[0]["Order"]
# Extract the tuple — parse from string like 'ARIMA(1, 1, 1)'
import re
nums = re.findall(r'\d+', best_order)
p, d, q = int(nums[0]), int(nums[1]), int(nums[2])

model = ARIMA(train["Births"], order=(p, d, q))
result = model.fit()
print(result.summary())

In [ ]:
# Forecast using get_forecast (returns prediction intervals)
forecast_obj = result.get_forecast(steps=HORIZON)
fc_mean = forecast_obj.predicted_mean
fc_summary = forecast_obj.summary_frame(alpha=0.05)

print("Forecast summary (first 10 steps):")
print(fc_summary.head(10))

In [ ]:
# Plot forecast vs actuals
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(train["Births"].iloc[-60:], label="Train (last 60 days)", color="black", alpha=0.5)
ax.plot(test["Births"], label="Actual (test)", color="tab:blue", linewidth=2)
ax.plot(fc_mean.index, fc_mean.values, label=f"ARIMA({p},{d},{q}) forecast",
        color="tab:red", linestyle="--")
ax.fill_between(
    fc_summary.index,
    fc_summary["mean_ci_lower"],
    fc_summary["mean_ci_upper"],
    color="tab:red", alpha=0.15, label="95% CI",
)
ax.axvline(test.index[0], color="grey", linestyle=":", alpha=0.7)
ax.set_title(f"ARIMA({p},{d},{q}) Forecast — Daily Births")
ax.set_ylabel("Births")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Evaluation metrics
actual = test["Births"].values
pred = fc_mean.values

mae = mean_absolute_error(actual, pred)
rmse = np.sqrt(mean_squared_error(actual, pred))
mape = np.mean(np.abs((actual - pred) / actual)) * 100

print(f"ARIMA({p},{d},{q}) Test-Set Accuracy:")
print(f"  MAE:  {mae:.2f}")
print(f"  RMSE: {rmse:.2f}")
print(f"  MAPE: {mape:.2f}%")

---
## 6. Residual Diagnostics

In [ ]:
residuals = result.resid

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Time plot
axes[0, 0].plot(residuals, linewidth=0.5)
axes[0, 0].axhline(0, color="red", linestyle="--", alpha=0.5)
axes[0, 0].set_title("Residuals over Time")

# Histogram
axes[0, 1].hist(residuals, bins=25, edgecolor="black", alpha=0.7, density=True)
axes[0, 1].set_title("Residual Distribution")

# ACF
plot_acf(residuals, lags=20, ax=axes[1, 0], title="ACF of Residuals")

# Q-Q plot
from scipy import stats
stats.probplot(residuals, dist="norm", plot=axes[1, 1])
axes[1, 1].set_title("Q-Q Plot")

plt.suptitle(f"ARIMA({p},{d},{q}) Residual Diagnostics", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Ljung-Box test
lb_test = acorr_ljungbox(residuals, lags=[10, 15, 20], return_df=True)
print("Ljung-Box test on ARIMA residuals:")
print(lb_test)
print()
print("p-values > 0.05 → residuals are consistent with white noise.")
print("The model has captured the autocorrelation structure adequately.")

---
## 7. Automatic Order Selection with `auto_arima`

Manually selecting $p$, $d$, $q$ by examining ACF/PACF is educational but
tedious in practice.  The `pmdarima` library provides `auto_arima`, which
performs a stepwise search over orders, minimising AIC (or BIC).

```python
from pmdarima import auto_arima
auto_model = auto_arima(train, seasonal=False, stepwise=True, trace=True)
```

In [ ]:
from pmdarima import auto_arima

auto_model = auto_arima(
    train["Births"],
    seasonal=False,
    stepwise=True,
    trace=True,
    suppress_warnings=True,
    information_criterion="aic",
)

print(f"\nBest model: {auto_model.order}")
print(auto_model.summary())

In [ ]:
# Forecast with auto_arima
auto_fc, auto_ci = auto_model.predict(n_periods=HORIZON, return_conf_int=True, alpha=0.05)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train["Births"].iloc[-60:], label="Train (last 60)", color="black", alpha=0.5)
ax.plot(test["Births"], label="Actual", color="tab:blue", linewidth=2)
ax.plot(test.index, auto_fc, label=f"auto_arima {auto_model.order}",
        color="tab:green", linestyle="--")
ax.fill_between(test.index, auto_ci[:, 0], auto_ci[:, 1],
                color="tab:green", alpha=0.15, label="95% CI")
ax.axvline(test.index[0], color="grey", linestyle=":", alpha=0.7)
ax.set_title(f"auto_arima({auto_model.order}) Forecast")
ax.set_ylabel("Births")
ax.legend()
plt.tight_layout()
plt.show()

auto_mae = mean_absolute_error(test["Births"].values, auto_fc)
auto_rmse = np.sqrt(mean_squared_error(test["Births"].values, auto_fc))
print(f"auto_arima MAE:  {auto_mae:.2f}")
print(f"auto_arima RMSE: {auto_rmse:.2f}")

---
## 8. Understanding ARIMA Special Cases

ARIMA generalises several models we already know:

| ARIMA order | Equivalent model |
|-------------|------------------|
| ARIMA(0,0,0) | White noise |
| ARIMA(1,0,0) | AR(1) |
| ARIMA(0,0,1) | MA(1) |
| ARIMA(0,1,0) | Random walk |
| ARIMA(0,1,0) with constant | Random walk with drift |
| ARIMA(1,1,0) | Differenced AR(1) |
| ARIMA(0,1,1) | Simple Exponential Smoothing (approximately) |

In [ ]:
# Demonstrate: ARIMA(0,1,0) = random walk
rw_model = ARIMA(train["Births"], order=(0, 1, 0)).fit()
rw_fc = rw_model.get_forecast(steps=HORIZON)

print("ARIMA(0,1,0) = Random Walk")
print(f"Forecast (constant): {rw_fc.predicted_mean.values[:5]}")
print(f"Last training value: {train['Births'].iloc[-1]}")
print("\nAll forecast values equal the last observation — exactly the naive method.")

In [ ]:
# Demonstrate: ARIMA(0,1,0) with trend = random walk with drift
rwd_model = ARIMA(train["Births"], order=(0, 1, 0), trend="t").fit()
rwd_fc = rwd_model.get_forecast(steps=HORIZON)

print("ARIMA(0,1,0) with trend = Random Walk with Drift")
print(f"Forecast (first 5): {rwd_fc.predicted_mean.values[:5].round(2)}")
print("\nForecasts increase/decrease linearly — similar to the drift method.")

---
## 9. Comparing Manual vs Automatic Selection

In [ ]:
# Summary comparison
comparison = pd.DataFrame({
    "Model": [
        f"Manual ARIMA({p},{d},{q})",
        f"auto_arima {auto_model.order}",
        "Naive (random walk)",
    ],
    "MAE": [
        mae,
        auto_mae,
        mean_absolute_error(test["Births"].values,
                            np.full(HORIZON, train["Births"].iloc[-1])),
    ],
})
comparison["MAE"] = comparison["MAE"].round(2)
comparison = comparison.sort_values("MAE").reset_index(drop=True)
print(comparison.to_string(index=False))
print(f"\nBest model: {comparison.iloc[0]['Model']}")

---
## 10. The `get_forecast` and `summary_frame` Pattern

The modern ARIMA API has a clean pattern for forecasting with confidence intervals:

```python
forecast_obj = result.get_forecast(steps=24)
forecast_obj.predicted_mean       # point forecasts
forecast_obj.conf_int(alpha=0.05) # confidence intervals
forecast_obj.summary_frame()      # DataFrame with mean, CI
```

This replaces the old-style `result.forecast()` and `result.predict(typ='levels')`
patterns from the deprecated API.

In [ ]:
# Demonstrate summary_frame
sf = result.get_forecast(steps=10).summary_frame(alpha=0.05)
print("Forecast summary_frame:")
print(sf.round(2))
print()
print("Columns: mean (point forecast), mean_se (standard error),")
print("         mean_ci_lower / mean_ci_upper (confidence interval).")

---
## 11. Choosing $d$: How Many Differences?

Rules of thumb:
- If ADF rejects (p < 0.05) and KPSS does not reject → $d = 0$ (already stationary)
- If ADF does not reject and KPSS rejects → $d \geq 1$
- If both reject or neither rejects → borderline; try $d = 0$ and $d = 1$

`pmdarima` has a helper function:

In [ ]:
from pmdarima.arima import ndiffs

d_adf = ndiffs(births["Births"], test="adf")
d_kpss = ndiffs(births["Births"], test="kpss")
d_pp = ndiffs(births["Births"], test="pp")

print("Estimated number of differences (d):")
print(f"  ADF test:  d = {d_adf}")
print(f"  KPSS test: d = {d_kpss}")
print(f"  PP test:   d = {d_pp}")

---
## 12. Prediction Intervals Widen Over Time

A key feature of ARIMA forecasts: prediction intervals widen with the
forecast horizon.  This reflects increasing uncertainty about the future.

For an ARIMA(0,1,0) (random walk), the prediction interval grows as
$\pm 1.96 \sigma \sqrt{h}$ — proportional to $\sqrt{h}$.

In [ ]:
# Show widening prediction intervals
long_fc = result.get_forecast(steps=60)
long_sf = long_fc.summary_frame(alpha=0.05)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train["Births"].iloc[-60:], label="Train", color="black", alpha=0.5)
ax.plot(long_sf["mean"], label="Forecast", color="tab:red")
ax.fill_between(long_sf.index, long_sf["mean_ci_lower"], long_sf["mean_ci_upper"],
                color="tab:red", alpha=0.15, label="95% CI")
ax.set_title("Prediction Intervals Widen with Horizon")
ax.set_ylabel("Births")
ax.legend()
plt.tight_layout()
plt.show()

print("Uncertainty grows with the forecast horizon.")
print("This is a natural and honest feature of statistical forecasting.")

---
## 13. In-Sample Fit Quality

In [ ]:
# In-sample fitted values
fitted = result.fittedvalues

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train["Births"], label="Actual", alpha=0.7)
ax.plot(fitted, label="Fitted", alpha=0.7, linestyle="--", color="tab:red")
ax.set_title(f"ARIMA({p},{d},{q}) — In-Sample Fit")
ax.set_ylabel("Births")
ax.legend()
plt.tight_layout()
plt.show()

in_sample_mae = mean_absolute_error(train["Births"].values[d:], fitted.values[d:])
print(f"In-sample MAE: {in_sample_mae:.2f}")

---
## Summary

- **ARIMA($p$,$d$,$q$)** = AR($p$) + differencing($d$) + MA($q$).
- In backshift notation: $\Phi(B)(1-B)^d y_t = c + \Theta(B)\varepsilon_t$.
- The **ARIMA modelling workflow**: plot → test stationarity → determine $d$ →
  ACF/PACF → tentative $p$, $q$ → fit → check residuals → forecast.
- Use the **modern API**: `from statsmodels.tsa.arima.model import ARIMA`.
- `result.get_forecast(steps=h).summary_frame()` gives forecasts with CIs.
- **`auto_arima`** from `pmdarima` automates order selection via AIC/BIC.
- ARIMA(0,1,0) = random walk; ARIMA(0,0,$q$) = MA($q$); ARIMA($p$,0,0) = AR($p$).
- Prediction intervals widen with the forecast horizon — this is expected.
- Next: seasonal ARIMA (SARIMA) for data with seasonal patterns.

In [ ]:
print("Next notebook: Seasonal ARIMA (SARIMA) — extending ARIMA to handle")
print("seasonal patterns like those in airline passengers and CO2 data.")